# World model starter — predict next state from current state

Load the trajectories split and train a tiny MLP that maps
`state_t -> state_t+1`. This is deliberately simple: the point is to
show the shape of the data, not to propose a state-of-the-art model.

In [ ]:
from datasets import load_dataset
import numpy as np, torch

ds = load_dataset('twelvedata/financial-world-model', 'trajectories_1day')
sample = ds['train'][0]
print('feature_names:', sample['feature_names'])
print('states shape:', np.array(sample['states']).shape)
print('next_states shape:', np.array(sample['next_states']).shape)

In [ ]:
def to_tensors(split):
    xs, ys = [], []
    for row in split:
        s = np.array(row['states'], dtype=np.float32)
        n = np.array(row['next_states'], dtype=np.float32)
        xs.append(s); ys.append(n)
    return np.concatenate(xs), np.concatenate(ys)

Xtr, Ytr = to_tensors(ds['train'].select(range(200)))  # small subset for demo
Xva, Yva = to_tensors(ds['val'].select(range(50)))
print(Xtr.shape, Ytr.shape)

In [ ]:
import torch, torch.nn as nn, torch.optim as optim

# Mask non-finite (NaN from warmup rows) so we don't pollute loss.
finite = np.isfinite(Xtr).all(axis=1) & np.isfinite(Ytr).all(axis=1)
Xtr, Ytr = Xtr[finite], Ytr[finite]
finite = np.isfinite(Xva).all(axis=1) & np.isfinite(Yva).all(axis=1)
Xva, Yva = Xva[finite], Yva[finite]

F = Xtr.shape[1]
model = nn.Sequential(nn.Linear(F, 128), nn.ReLU(), nn.Linear(128, F))
opt = optim.Adam(model.parameters(), lr=1e-3)
xt, yt = torch.tensor(Xtr), torch.tensor(Ytr)
xv, yv = torch.tensor(Xva), torch.tensor(Yva)
for epoch in range(5):
    pred = model(xt)
    loss = ((pred - yt) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        val_loss = ((model(xv) - yv) ** 2).mean().item()
    print(f'epoch {epoch}  train={loss.item():.4f}  val={val_loss:.4f}')